# Day 5 — blind human rating

The secondary question of the whole project: **does replacement score track whether a human can
actually extract a mechanism from the graph?**

Pre-registered in D3.8:

- 40 graphs, drawn 5 from each of 8 quantile bins of replacement score
- 10 re-inserted as unmarked duplicates → **50 presentations** in randomised order
- Rated **blind to every metric**, 3-point rubric, 5-minute cap per graph
- Intra-rater agreement on the duplicates reported as quadratic weighted kappa; below 0.6, P5 is
  reported as uninterpretable

**The rubric, fixed before any graph was viewed:**

| | |
|---|---|
| **2** | I can state a causal path from input tokens to the output in one sentence, naming the intermediate features |
| **1** | I can identify at least one meaningful intermediate feature, but cannot trace a complete path |
| **0** | Neither |

### Blinding

This notebook never prints a replacement score, a completeness score or a bin index until every one
of the 50 presentations has been rated. Selection uses the scores; display does not. Ratings are
written to disk as you go, so the session can be resumed.

**The one thing that would ruin the day:** opening `results.csv` in another tab. Don't.

## Part 0 — Install

In [1]:
!pip install -q circuit-tracer
print("installed")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 153.2/153.2 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 15.9 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 272.5/272.5 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 968.6/968.6 kB 25.5 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 33.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.5/82.5 kB 3.5 MB/s eta 0:00:00
installed


## Part 1 — Environment and inputs

In [ ]:
import os
os.environ["HF_HOME"] = "/kaggle/working/hf"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import sys, time, gc, json, glob, shutil, random, datetime
import numpy as np, pandas as pd, torch

OUT = "/kaggle/working/out"
GRAPHS = "/kaggle/working/rating_graphs"
os.makedirs(OUT, exist_ok=True); os.makedirs(GRAPHS, exist_ok=True)

def find_input(name, required=True):
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True) + glob.glob(f"/kaggle/working/{name}")
    if not hits and required:
        raise FileNotFoundError(f"{name} not found. Add Input > Upload a dataset containing it.")
    return hits[0] if hits else None

def free():
    gc.collect(); torch.cuda.empty_cache()

src = find_input("ct_utils.py")
if os.path.abspath(src) != "/kaggle/working/ct_utils.py":
    shutil.copy(src, "/kaggle/working/ct_utils.py")
sys.path.insert(0, "/kaggle/working")
import importlib, ct_utils; importlib.reload(ct_utils)
from ct_utils import to_np

from kaggle_secrets import UserSecretsClient
_t = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["HF_TOKEN"] = _t; os.environ["HUGGING_FACE_HUB_TOKEN"] = _t
from huggingface_hub import login; login(token=_t)
print("ready")

## Part 2 — Blind selection

Reads `results.csv` to build the sample, then **discards the scores from memory**. The manifest
written to disk holds only presentation order, prompt id, prompt text and category. Bin
assignments go to a separate sealed file that is not read again until Part 5.

In [ ]:
results = pd.read_csv(find_input("results.csv"), keep_default_na=False)
corpus = pd.read_csv(find_input("corpus.csv"),
                     dtype={"expected": str, "subtype": str, "prompt": str}, keep_default_na=False)

ok = results[results.status == "ok"].copy()
ok["replacement_score"] = ok.replacement_score.astype(float)
assert len(ok) >= 40, f"only {len(ok)} usable graphs"

# 8 quantile bins of replacement score, 5 graphs per bin, fixed seed
ok["bin"] = pd.qcut(ok.replacement_score, 8, labels=False, duplicates="drop")
sel = ok.groupby("bin", group_keys=False).sample(5, random_state=0)[["prompt_id", "bin"]]
assert len(sel) == 40, f"selected {len(sel)}, expected 40"

rng = random.Random(0)
dup_ids = rng.sample(list(sel.prompt_id), 10)
presentations = list(sel.prompt_id) + dup_ids
rng.shuffle(presentations)

manifest = pd.DataFrame({"presentation": range(1, len(presentations) + 1),
                         "prompt_id": presentations})
manifest = manifest.merge(corpus[["prompt_id", "category", "subtype", "prompt", "n_tok"]],
                          on="prompt_id", how="left")
manifest.to_csv(f"{OUT}/rating_manifest.csv", index=False)

# SEALED: not opened again until Part 5
sel.assign(is_duplicate_source=sel.prompt_id.isin(dup_ids)).to_csv(f"{OUT}/rating_key_SEALED.csv",
                                                                   index=False)
del ok, results, sel
free()
print(f"{len(manifest)} presentations, {manifest.prompt_id.nunique()} unique graphs")
print(manifest.groupby('category').size().to_string())
print("\nscores discarded from memory; manifest holds no metrics")

## Part 3 — Generate the graphs

Each unique prompt is re-attributed and its pruned graph written to disk, so the rating session
does not wait on the GPU. ~40 graphs at ~18 s is about 12 minutes. Resumable.

Re-attribution differs from the corpus run by ~1e-4 in replacement score (Day 3, D3.5). Irrelevant
here: you are judging whether a mechanism is traceable, not reading a number.

In [ ]:
from circuit_tracer import ReplacementModel, attribute
import circuit_tracer.graph as ctg

cfg = json.load(open(find_input("day3_config.json")))
AK = cfg["attr_kwargs"]

model = ReplacementModel.from_pretrained(
    "google/gemma-2-2b", "gemma", backend="nnsight",
    dtype=torch.bfloat16, device="cuda", lazy_encoder=True)
print("model loaded")

In [ ]:
# What does the library offer for producing a viewable graph?
import inspect
from circuit_tracer.utils import create_graph_files
print(inspect.signature(create_graph_files))
print(inspect.getdoc(create_graph_files)[:1200])

In [ ]:
PRUNE_THRESHOLD = 0.8      # D3.8: graphs are viewed pruned at 0.8

todo = [p for p in manifest.prompt_id.unique()
        if not os.path.exists(f"{GRAPHS}/{p}.pt")]
print(f"{len(todo)} graphs to generate")

for i, pid in enumerate(todo, 1):
    row = corpus[corpus.prompt_id == pid].iloc[0]
    t0 = time.time()
    g = attribute(prompt=row.prompt, model=model, **AK)
    g.to_pt(f"{GRAPHS}/{pid}.pt")
    del g; free()
    print(f"{i:2d}/{len(todo)}  {pid}  {time.time()-t0:5.1f}s")
    if i % 5 == 0:
        print("  disk:", os.popen("df -BG /kaggle/working | awk 'NR==2{print $4}'").read().strip())

print("\nNOTE: a large graph is ~685 MB. If disk runs low, generate and rate in batches -")
print("delete each .pt after rating it, or reduce to the graphs not yet presented.")

### Viewing a graph

Rating "can I trace a mechanism" needs an actual graph view, not a score. Two routes — **verify
one before rating all 50**:

1. **Neuronpedia's circuit viewer.** `create_graph_files` writes the JSON its frontend expects.
   Upload one and check it renders, then use it for all of them.
2. **The in-notebook fallback below**, which lists the surviving nodes by influence with their
   layer, token position and strongest incoming edges. Cruder, but self-contained.

Whichever you choose, **use the same one for all 50 presentations**. Switching part-way makes the
ratings incomparable, and the duplicates would measure the viewer rather than you.

In [ ]:
from circuit_tracer import Graph

def node_table(pid, top_n=25):
    '''Pruned graph as a table: which nodes survive, where they sit, what feeds them.'''
    g = Graph.from_pt(f"{GRAPHS}/{pid}.pt")
    n_feat = len(g.selected_features)
    nt = len(to_np(g.input_tokens))
    nl = cfg["n_layers"]
    toks = [model.tokenizer.decode([int(t)]) for t in to_np(g.input_tokens).astype(int)]

    A = g.adjacency_matrix.float().abs()
    An = A / A.sum(1, keepdim=True).clamp_min(1e-30)
    w = torch.zeros(An.shape[0], device=An.device)
    w[-len(g.logit_probabilities):] = g.logit_probabilities.float().to(An.device)
    v, infl = w.clone(), torch.zeros_like(w)
    while float(v.sum()) > 1e-12:
        v = v @ An; infl += v
    inf = infl.cpu().numpy()

    def label(i):
        if i < n_feat:
            meta = to_np(g.selected_features)[i]
            meta = np.atleast_1d(meta)
            if meta.size >= 3:
                lay, pos, fid = int(meta[0]), int(meta[1]), int(meta[2])
                return f"feature L{lay:02d} pos{pos} ({toks[pos]!r}) #{fid}"
            return f"feature #{int(meta[0])}"
        if i < n_feat + nl * nt:
            j = i - n_feat
            return f"ERROR L{j // nt:02d} pos{j % nt} ({toks[j % nt]!r})"
        if i < n_feat + nl * nt + nt:
            p = i - n_feat - nl * nt
            return f"token pos{p} {toks[p]!r}"
        return "LOGIT"

    order = np.argsort(-inf)[:top_n]
    print(f"prompt: {corpus.loc[corpus.prompt_id == pid, 'prompt'].iloc[0]!r}")
    print(f"tokens: {list(enumerate(toks))}")
    print(f"nodes {An.shape[0]}, features {n_feat}\n")
    print(f"{'influence':>10}  node")
    for i in order:
        srcs = np.argsort(-An[i].cpu().numpy())[:3]
        feeds = ", ".join(label(int(s)) for s in srcs if An[i, int(s)] > 0)
        print(f"{inf[i]:10.4f}  {label(int(i))}")
        if feeds:
            print(f"{'':10}    <- {feeds}")
    del g, A, An, w, v, infl; free()

node_table(manifest.prompt_id.iloc[0])

In [ ]:
# Optional: Neuronpedia-format files for the same graph, to check route 1 works.
try:
    create_graph_files(graph_or_path=f"{GRAPHS}/{manifest.prompt_id.iloc[0]}.pt",
                       slug=f"rating-{manifest.prompt_id.iloc[0]}",
                       output_path=f"{GRAPHS}/np_files",
                       node_threshold=PRUNE_THRESHOLD)
    print(os.listdir(f"{GRAPHS}/np_files"))
except TypeError as e:
    print("signature differs - check the printout above and adjust:", e)

## Part 4 — Rate

`show(n)` displays presentation *n*, `record(rating, note)` saves it. Ratings append to
`ratings.jsonl` immediately, so a dropped session costs at most one item.

Work through 1 to 50 in order. **5-minute cap per graph** — the timer warns you. Don't go back and
revise: a first-pass judgement made consistently is worth more than a considered one made
inconsistently, and revisions would contaminate the duplicate check.

In [ ]:
RATINGS = f"{OUT}/ratings.jsonl"
_current = {}

def rated():
    if not os.path.exists(RATINGS):
        return set()
    return {json.loads(l)["presentation"] for l in open(RATINGS) if l.strip()}

def show(n=None):
    global _current
    done = rated()
    if n is None:
        left = [p for p in manifest.presentation if p not in done]
        if not left:
            print("all 50 rated"); return
        n = left[0]
    row = manifest[manifest.presentation == n].iloc[0]
    _current = dict(presentation=int(n), prompt_id=row.prompt_id, t0=time.time())
    print(f"=== presentation {n} of {len(manifest)}   ({len(done)} rated) ===\n")
    node_table(row.prompt_id)
    print("\n2 = full causal path | 1 = some feature, no path | 0 = neither")
    print("then: record(rating, 'optional note')")

def record(rating, note=""):
    assert rating in (0, 1, 2), "rating must be 0, 1 or 2"
    assert _current, "call show() first"
    secs = time.time() - _current["t0"]
    if secs > 300:
        print(f"note: {secs/60:.1f} min, over the 5-minute cap")
    rec = dict(presentation=_current["presentation"], prompt_id=_current["prompt_id"],
               rating=int(rating), note=note, seconds=round(secs, 1),
               at=datetime.datetime.now().isoformat(timespec="seconds"))
    with open(RATINGS, "a") as f:
        f.write(json.dumps(rec) + "\n"); f.flush(); os.fsync(f.fileno())
    print(f"recorded {rating} ({secs:.0f}s). {len(rated())}/{len(manifest)} done. show() for next.")

print(f"{len(rated())}/{len(manifest)} already rated. Run show() to begin.")

In [ ]:
show()

## Part 5 — Unblind

**Only run this once all 50 are rated.** It asserts completeness first.

Two outputs: intra-rater agreement on the 10 duplicates, and — the point of the whole exercise —
the relationship between replacement score and whether a human could read the graph.

In [ ]:
rt = pd.DataFrame([json.loads(l) for l in open(RATINGS) if l.strip()])
assert len(rt) == len(manifest), f"only {len(rt)}/{len(manifest)} rated"
assert rt.presentation.is_unique

# --- intra-rater reliability on the duplicates ---
dups = rt.groupby("prompt_id").filter(lambda d: len(d) == 2)
pairs = dups.sort_values("presentation").groupby("prompt_id").rating.apply(list)
a = np.array([p[0] for p in pairs]); b = np.array([p[1] for p in pairs])
print(f"{len(a)} duplicate pairs")
print(pd.DataFrame({"first": a, "second": b}).to_string(index=False))

def quadratic_kappa(x, y, k=3):
    # Kappa with quadratic weights: disagreeing by 2 counts 4x worse than disagreeing by 1.
    n = len(x)
    O = np.zeros((k, k))
    for i, j in zip(x, y):
        O[i, j] += 1
    W = np.array([[(i - j) ** 2 / (k - 1) ** 2 for j in range(k)] for i in range(k)])
    E = np.outer(np.bincount(x, minlength=k), np.bincount(y, minlength=k)) / n
    den = (W * E).sum()
    return float(1 - (W * O).sum() / den) if den > 0 else float("nan")

KAPPA = quadratic_kappa(a, b)
print(f"\nexact agreement: {(a == b).mean():.0%}")
print(f"quadratic weighted kappa: {KAPPA:.3f}   (P5 uninterpretable if < 0.6)")

In [ ]:
# --- P5: does replacement score track readability? ---
from scipy.stats import spearmanr

first = rt.sort_values("presentation").drop_duplicates("prompt_id")     # first presentation only
res = pd.read_csv(find_input("results.csv"), keep_default_na=False)
key = pd.read_csv(f"{OUT}/rating_key_SEALED.csv")
j = first.merge(res[["prompt_id", "replacement_score", "completeness_score", "err_pos_gini_norm",
                     "node_count_curve", "category"]], on="prompt_id")
j["replacement_score"] = j.replacement_score.astype(float)

rho, p = spearmanr(j.replacement_score, j.rating)
print(f"P5 -- Spearman(replacement, rating) = {rho:+.3f}, p = {p:.3f}, n = {len(j)}")
print(f"pre-registered prediction: positive but weak, |rho| < 0.4  ->  "
      f"{'HOLDS' if abs(rho) < 0.4 else 'FAILS'}")
print()
print(j.groupby("rating").agg(n=("rating", "size"),
                              mean_replacement=("replacement_score", "mean"),
                              mean_gini=("err_pos_gini_norm", "mean")).round(3).to_string())
print()
print(pd.crosstab(j.category, j.rating).to_string())

j.to_csv(f"{OUT}/ratings_unblinded.csv", index=False)
json.dump(dict(kappa=float(KAPPA), spearman_replacement=float(rho), p_value=float(p),
               n_rated=int(len(rt)), n_unique=int(len(j)),
               rating_counts=rt.rating.value_counts().to_dict(),
               median_seconds=float(rt.seconds.median())),
          open(f"{OUT}/day5_summary.json", "w"), indent=2)
print("\nsaved ratings_unblinded.csv and day5_summary.json -- DOWNLOAD /kaggle/working/out/")

## Record in the pre-registration

> **D3.8 — outcome [DATE].** 50 presentations (40 unique graphs, 10 unmarked duplicates) rated
> blind to all metrics, in randomised order, median [__] s per graph. Rating distribution:
> [__]. Exact agreement on duplicates [__]%; quadratic weighted kappa [__]
> (threshold 0.6 → P5 [interpretable / uninterpretable]).
>
> **P5 — outcome.** Spearman correlation between replacement score and rating: [__] (p = [__],
> n = 40). Prediction was positive but weak, |rho| < 0.4 → [HOLDS / FAILS].
>
> **Limitations as stated in 6.5:** single rater, not blind to the project's hypotheses, and the
> graphs were viewed via [route]. Graphs were re-attributed for rating and differ from the corpus
> run by ~1e-4 in replacement score, which does not affect a three-point readability judgement.